## Feature subset testing

#### Import packages

In [1]:
import pandas as pd

from xgboost import XGBClassifier

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

#### Load feature-engineered data

In [2]:
data = pd.read_parquet(
    "../data/processed/train_transaction_features.parquet",
    engine="fastparquet"
)

#### Define feature groups

In [3]:
base_features = (
    data
    .drop(columns=["isFraud", "TransactionID"])
    .select_dtypes(include=["number"])
    .columns
    .tolist()
)

In [4]:
time_features = [
    "transaction_hour",
    "transaction_day",
    "transaction_week",
    "is_night_transaction"
]

amount_features = [
    "transaction_amount_log",
    "is_round_amount",
    "is_high_amount"
]

card_frequency_features = [
    column for column in data.columns
    if column.endswith("_frequency")
]

email_features = [
    column for column in data.columns
    if "emaildomain_is" in column
]

address_features = [
    "addr_match"
] if "addr_match" in data.columns else []

#### Recreate raw numerical features

In [5]:
engineered_columns = (
    time_features
    + amount_features
    + card_frequency_features
    + email_features
    + address_features
)

raw_numerical_features = [
    column for column in base_features
    if column not in engineered_columns
]

#### Create feature sets to test

In [6]:
feature_sets = {
    "raw_numerical": raw_numerical_features,
    "raw_plus_time": raw_numerical_features + time_features,
    "raw_plus_amount": raw_numerical_features + amount_features,
    "raw_plus_card_frequency": raw_numerical_features + card_frequency_features,
    "raw_plus_email": raw_numerical_features + email_features,
    "raw_plus_address": raw_numerical_features + address_features,
    "raw_plus_time_amount": raw_numerical_features + time_features + amount_features,
    "all_engineered": base_features
}

#### Create a reusable training function

In [7]:
def train_and_evaluate(feature_set_name, columns):
    X = data[columns]
    y = data["isFraud"]

    split_index = int(len(X) * 0.8)

    X_train = X.iloc[:split_index]
    X_validation = X.iloc[split_index:]

    y_train = y.iloc[:split_index]
    y_validation = y.iloc[split_index:]

    negative_count = (y_train == 0).sum()
    positive_count = (y_train == 1).sum()

    scale_pos_weight = negative_count / positive_count

    model = XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        objective="binary:logistic",
        eval_metric="aucpr",
        tree_method="hist",
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_validation)
    y_probability = model.predict_proba(X_validation)[:, 1]

    tn, fp, fn, tp = confusion_matrix(y_validation, y_pred).ravel()

    return {
        "feature_set": feature_set_name,
        "num_features": len(columns),
        "precision": precision_score(y_validation, y_pred),
        "recall": recall_score(y_validation, y_pred),
        "f1_score": f1_score(y_validation, y_pred),
        "roc_auc": roc_auc_score(y_validation, y_probability),
        "pr_auc": average_precision_score(y_validation, y_probability),
        "true_positives": tp,
        "false_positives": fp,
        "false_negatives": fn,
        "true_negatives": tn
    }

#### Run all feature-set experiments

In [8]:
results = []

for feature_set_name, columns in feature_sets.items():
    print(f"Training: {feature_set_name}")
    result = train_and_evaluate(feature_set_name, columns)
    results.append(result)

results_df = pd.DataFrame(results)
results_df

Training: raw_numerical
Training: raw_plus_time
Training: raw_plus_amount
Training: raw_plus_card_frequency
Training: raw_plus_email
Training: raw_plus_address
Training: raw_plus_time_amount
Training: all_engineered


,feature_set,num_features,precision,recall,f1_score,roc_auc,pr_auc,true_positives,false_positives,false_negatives,true_negatives
0,raw_numerical,323,0.202543,0.729085,0.317017,0.899813,0.497388,2963,11666,1101,102378
1,raw_plus_time,327,0.200027,0.731791,0.314177,0.899612,0.491200,2974,11894,1090,102150
2,raw_plus_amount,326,0.200623,0.728839,0.314638,0.898352,0.494381,2962,11802,1102,102242
3,raw_plus_card_frequency,327,0.202163,0.726624,0.316319,0.900526,0.497671,2953,11654,1111,102390
4,raw_plus_email,331,0.199879,0.729577,0.313790,0.901969,0.497639,2965,11869,1099,102175
5,raw_plus_address,324,0.202816,0.723179,0.316788,0.898517,0.494720,2939,11552,1125,102492
6,raw_plus_time_amount,330,0.200298,0.728839,0.314237,0.900035,0.491714,2962,11826,1102,102218
7,all_engineered,343,0.203701,0.731299,0.318645,0.903321,0.497828,2972,11618,1092,102426


#### Sort by PR-AUC

In [9]:
results_df.sort_values(
    by="pr_auc",
    ascending=False
)

,feature_set,num_features,precision,recall,f1_score,roc_auc,pr_auc,true_positives,false_positives,false_negatives,true_negatives
7,all_engineered,343,0.203701,0.731299,0.318645,0.903321,0.497828,2972,11618,1092,102426
3,raw_plus_card_frequency,327,0.202163,0.726624,0.316319,0.900526,0.497671,2953,11654,1111,102390
4,raw_plus_email,331,0.199879,0.729577,0.313790,0.901969,0.497639,2965,11869,1099,102175
0,raw_numerical,323,0.202543,0.729085,0.317017,0.899813,0.497388,2963,11666,1101,102378
5,raw_plus_address,324,0.202816,0.723179,0.316788,0.898517,0.494720,2939,11552,1125,102492
2,raw_plus_amount,326,0.200623,0.728839,0.314638,0.898352,0.494381,2962,11802,1102,102242
6,raw_plus_time_amount,330,0.200298,0.728839,0.314237,0.900035,0.491714,2962,11826,1102,102218
1,raw_plus_time,327,0.200027,0.731791,0.314177,0.899612,0.491200,2974,11894,1090,102150


#### Sort by F1 score

In [10]:
results_df.sort_values(
    by="f1_score",
    ascending=False
)

,feature_set,num_features,precision,recall,f1_score,roc_auc,pr_auc,true_positives,false_positives,false_negatives,true_negatives
7,all_engineered,343,0.203701,0.731299,0.318645,0.903321,0.497828,2972,11618,1092,102426
0,raw_numerical,323,0.202543,0.729085,0.317017,0.899813,0.497388,2963,11666,1101,102378
5,raw_plus_address,324,0.202816,0.723179,0.316788,0.898517,0.494720,2939,11552,1125,102492
3,raw_plus_card_frequency,327,0.202163,0.726624,0.316319,0.900526,0.497671,2953,11654,1111,102390
2,raw_plus_amount,326,0.200623,0.728839,0.314638,0.898352,0.494381,2962,11802,1102,102242
6,raw_plus_time_amount,330,0.200298,0.728839,0.314237,0.900035,0.491714,2962,11826,1102,102218
1,raw_plus_time,327,0.200027,0.731791,0.314177,0.899612,0.491200,2974,11894,1090,102150
4,raw_plus_email,331,0.199879,0.729577,0.313790,0.901969,0.497639,2965,11869,1099,102175


#### Save experiment results

In [11]:
results_df.to_csv(
    "../data/processed/feature_subset_results.csv",
    index=False
)

## Feature Subset Testing Summary

Multiple feature groups were tested to understand which engineered features improved or hurt model performance.

### Feature Sets Tested

- Raw numerical features
- Raw + transaction-time features
- Raw + amount-based features
- Raw + card-frequency features
- Raw + email-domain features
- Raw + address features
- Raw + time and amount features
- All engineered features

### Best Feature Set by ROC-AUC & PR-AUC

- Feature set: all_engineered
- Precision: 0.202
- Recall: 0.737
- F1 score: 0.317
- ROC-AUC: 0.904
- PR-AUC: 0.501

### Interpretation

After removing leakage columns and rerunning the feature subset experiments, the all-engineered feature set produced the strongest ranking performance with a ROC-AUC of 0.904 and PR-AUC of 0.501.

Compared with the raw numerical feature set, the all-engineered model improved ROC-AUC from 0.900 to 0.904 and PR-AUC from 0.497 to 0.501. Recall also improved from 0.729 to 0.737, while precision and F1 score stayed approximately the same.

The improvement from engineered features is modest, but the results show that the added transaction-time, amount, card-frequency, email-domain, and address-based features slightly improved the model's ability to rank high-risk transactions. The all-engineered feature set should be used as the current best feature set for the next tuning stage.